# Lab 06: How to build an evaluation set for RAG

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Write a test set for your pipeline, score it one question at a time, and report each kind of question on its own. The
moment that teaches it: the standard word-overlap score gives half credit to a wrong number of minutes and little
credit to a right answer written as a full sentence. And the pipeline looks good on the whole set while failing some
of the policy questions written to test it, one of them answering a Boreal question from the Atlas policy.

Lab 04 named the failing stage for a few wrong answers, one trace at a time, and Lab 05 rebuilt the same pipeline
with LangChain. Before you change a stage, you need to measure every question the same way each time, so you can
see what a change fixes and what it breaks: this lab writes that question set and scores the pipeline on it.

## What you will learn

- How to compute hit@3 and mean reciprocal rank (MRR) with every question in the denominator.
- What exact match (EM) and token F1 measure, and where they mislead on generated answers.
- How to write golden records with a type, known support, and references or an expected refusal, and how to version
  a set with hashes.
- How to run a pipeline on a golden set and keep one row per question.
- How to score refusals on their own, and report each kind of question as its own slice.

**In class:** Parts A to D. Parts E and F are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-18
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` from `models.env`. It embeds the
corpus once (about 21,000 tokens) and makes about 65 short chat calls, well under one US cent on `gpt-4o-mini`. The
saved outputs and every count in the text come from `gpt-4o-mini`; the README says what changed with `gpt-6-luna`. Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

An **evaluation set**, or **golden set**, is a fixed list of questions, each with what a right answer must contain
or that the system must refuse, written before the system runs and kept outside the index it searches. Every change to
the pipeline is then measured on the same questions.

---

## Part A: Score a small example by hand

Part A works each measure on numbers small enough to check in your head.

Retrieval first. A question's **known support** is the source its reference answer comes from. **hit@3** asks
whether a known source is among the top three results. The **reciprocal rank** is 1 divided by the rank of the first
known source, and 0 when none is in the top three; **MRR@3**, the mean reciprocal rank, is its average over the
questions, so it rewards finding the support early. Take three questions whose first support comes at rank 1, at
rank 3 and not at all:

In [3]:
ranks = [1, 3, None]          # rank of the first known source in the top 3; None: not in the top 3
reciprocal = [1 / rank if rank else 0 for rank in ranks]
found_any = sum(rank is not None for rank in ranks)
print("reciprocal ranks:", [round(r, 3) for r in reciprocal])
print(f"hit@3: {found_any} of {len(ranks)}")
print(f"MRR@3: {sum(reciprocal) / len(ranks):.3f}  (divided by all {len(ranks)} questions)")
print(f"MRR@3: {sum(reciprocal) / found_any:.3f}  (divided by the {found_any} that found something)")

reciprocal ranks: [1.0, 0.333, 0]
hit@3: 2 of 3
MRR@3: 0.444  (divided by all 3 questions)
MRR@3: 0.667  (divided by the 2 that found something)


Both lines run, and only the first is MRR: 0.444. Dividing by the two questions that found their support gives
0.667, and the miss disappears. Keep every question you asked in the denominator, and print it beside the result.

Now the answers. **Exact match (EM)** is 1 when the answer and the reference are the same text after normalising, and
0 otherwise. **Token F1** gives partial credit for the words the two share. Both come from SQuAD (2016), a reading
benchmark whose answers are short spans copied from a paragraph, the dataset our 30 development questions come from.
Normalise first, as SQuAD's scoring does: lower case, no punctuation, no *a*, *an* or *the*:

In [4]:
import re
import string

def normalize(text):
    text = "".join(c for c in text.lower() if c not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", text).split())

print(normalize("About 30 minutes."), "|", normalize("The Denver Broncos"))

about 30 minutes | denver broncos


Now F1 by hand for the answer *about 30 minutes* against the reference *30 minutes*. **Precision** is the share of
the answer's words that are in the reference; **recall** is the share of the reference's words that are in the
answer; F1 is their harmonic mean. `Counter(a) & Counter(b)` keeps each shared word as many times as it appears in
both:

In [5]:
from collections import Counter

answer_words, reference_words = normalize("about 30 minutes").split(), normalize("30 minutes").split()
overlap = sum((Counter(answer_words) & Counter(reference_words)).values())
precision, recall = overlap / len(answer_words), overlap / len(reference_words)
f1 = 2 * precision * recall / (precision + recall)
print(f"shared {overlap} | precision {precision:.3f} | recall {recall:.3f} | F1 {f1:.3f}"
      f" | EM {int(answer_words == reference_words)}")

shared 2 | precision 0.667 | recall 1.000 | F1 0.800 | EM 0


Two of the three answer words are in the reference, and both reference words are in the answer: F1 0.800, EM 0.

### Checkpoint A

| You should see | Our run |
|---|---|
| Ranks 1, 3 and none | hit@3 2 of 3; MRR@3 0.444, or 0.667 when the miss leaves the denominator |
| Normalised | `about 30 minutes \| denver broncos` |
| *about 30 minutes* against *30 minutes* | shared 2, precision 0.667, recall 1.000, F1 0.800, EM 0 |

---

## Part B: Write the measures as functions

Part B makes the measures reusable and checks them against Part A's numbers before any model answer is scored.

A question can have more than one accepted answer: SQuAD keeps the strings several annotators chose, such as *Miller*
and *Von Miller*. `em_f1()` scores the answer against each reference and keeps the best EM and the best F1, so a right
answer does not lose credit because one annotator wrote it differently. `first_rank()` returns the rank of the first
retrieved source that is in the known support, or `None`:

In [6]:
def em_f1(prediction, references):
    best_em, best_f1 = 0, 0.0
    for reference in references:
        a, b = normalize(prediction).split(), normalize(reference).split()
        overlap = sum((Counter(a) & Counter(b)).values())
        f1 = 0.0
        if overlap:
            precision, recall = overlap / len(a), overlap / len(b)
            f1 = 2 * precision * recall / (precision + recall)
        best_em, best_f1 = max(best_em, int(a == b)), max(best_f1, f1)
    return best_em, best_f1

def first_rank(sources, support):
    return next((rank for rank, source in enumerate(sources, 1) if source in support), None)

print(em_f1("about 30 minutes", ["30 minutes"]), "|", first_rank(["H04", "H01", "H02"], ["H02"]))

(0, 0.8) | 3


The same 0.800 as by hand, and rank 3 for H02 in third place. The `if overlap` test comes before any division, so an
empty answer scores 0 instead of raising an error (see Troubleshooting).

Four answers to score.

*Predict first:* which of the four answers is wrong, and which one gets the lowest F1?

In [7]:
cases = [
    ("30 minutes", ["30 minutes"]),
    ("30 minutes", ["10 minutes"]),
    ("Nikola Tesla was born on July 10, 1856.", ["1856"]),
    ("about 30 minutes", ["30 minutes", "half an hour"]),
]
for prediction, references in cases:
    em, f1 = em_f1(prediction, references)
    print(f"EM {em} | F1 {f1:.3f} | {prediction!r} against {references}")

EM 1 | F1 1.000 | '30 minutes' against ['30 minutes']
EM 0 | F1 0.500 | '30 minutes' against ['10 minutes']
EM 0 | F1 0.222 | 'Nikola Tesla was born on July 10, 1856.' against ['1856']
EM 0 | F1 0.800 | 'about 30 minutes' against ['30 minutes', 'half an hour']


Read the second and third lines together. *30 minutes*, when the reference says *10 minutes*, is wrong, and F1 gives
it 0.500: one of its two words matches. The Tesla sentence is right, and F1 gives it 0.222: it has seven words the
reference lacks. F1 measures shared words, not whether the fact is right. The last line keeps the better of two
references, 0.800, not their average.

### Checkpoint B

| You should see | Our run |
|---|---|
| The functions | `(0, 0.8) \| 3` |
| Four answers | EM 1, F1 1.000; EM 0, F1 0.500 (wrong); EM 0, F1 0.222 (right); EM 0, F1 0.800 |

---

## Part C: Write ten golden records

Part C writes the questions the pipeline will be scored on, with what counts as right for each one.

A **golden record** holds one question and what a right answer must do, so that any run can be scored without a
person reading every answer. Each record here has:

- `question_id` and `split`: a stable ID, and `development` (questions you may look at while you change the pipeline);
- `question` and `type`: the text, and the kind of question (answerable, paraphrase, exception, ...);
- `known_support`: the sources a reference answer comes from, as source IDs (`S03-P00`), not chunk IDs;
- `references`, the accepted answers; or none and `refuse: True` when the right reply is *Insufficient evidence.*

A development question already holds most of this, as the data file shows:

In [8]:
import json

DATA = ROOT / "data"
dev = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
tesla = next(q for q in dev if q["question"].startswith("In what year was Nikola Tesla born"))
print(json.dumps(tesla, ensure_ascii=False))

{"question_id": "56df9e2838dc4217001520f6", "question": "In what year was Nikola Tesla born? ", "answers": [{"answer_start": 54, "text": "1856"}, {"answer_start": 54, "text": "1856"}, {"answer_start": 54, "text": "1856"}], "known_support": ["S03-P00"], "article_index": 3}


Three annotators chose the same answer, *1856*. Build the record by hand: its ID, text and `known_support`, and the
answer strings without repeats:

In [9]:
record = {"question_id": tesla["question_id"], "split": "development", "question": tesla["question"],
          "type": "answerable", "known_support": tesla["known_support"],
          "references": sorted({a["text"] for a in tesla["answers"]}), "refuse": False}
print(record)

{'question_id': '56df9e2838dc4217001520f6', 'split': 'development', 'question': 'In what year was Nikola Tesla born? ', 'type': 'answerable', 'known_support': ['S03-P00'], 'references': ['1856'], 'refuse': False}


`record_from_dev()` does the same for any development question. Choose seven that differ: the Tesla year, a
question answered by a single year, Lab 01's hit@1 miss (the Super Bowl MVP), one with two accepted answers (*8* and
*eight*), one with a long reference (the Vistula's animals), one with two different references (the meaning of
*Norman*), and the complexity question Lab 04 found the pipeline answers wrongly:

In [10]:
def record_from_dev(question):
    return {"question_id": question["question_id"], "split": "development", "question": question["question"],
            "type": "answerable", "known_support": question["known_support"],
            "references": sorted({a["text"] for a in question["answers"]}), "refuse": False}

chosen = ["In what year was Nikola Tesla born", "In what year did Nikola Tesla emigrate", "Who was the Super Bowl 50 MVP",
          "How many appearances have the Denver Broncos", "What animals does the Vistula",
          "What is the original meaning of the word Norman", "What measure of a computational problem"]
records = [record_from_dev(q) for start in chosen for q in dev if q["question"].startswith(start)]
for r in records:
    print(r["known_support"], r["references"], "|", r["question"])

['S03-P00'] ['1856'] | In what year was Nikola Tesla born? 
['S03-P01'] ['1884'] | In what year did Nikola Tesla emigrate to the United States?
['S00-P02'] ['Miller', 'Von Miller'] | Who was the Super Bowl 50 MVP?
['S00-P01'] ['8', 'eight'] | How many appearances have the Denver Broncos made in the Super Bowl?
['S01-P02'] ['otter, beaver and hundreds of bird species', 'otter, beaver and hundreds of bird species.'] | What animals does the Vistula river's ecosystem include?
['S02-P02'] ['Norseman, Viking', 'Viking'] | What is the original meaning of the word Norman?
['S04-P01'] ['if its solution requires significant resources', 'its solution requires significant resources'] | What measure of a computational problem broadly defines the inherent difficulty of the solution?


The development questions are all answerable: each has a paragraph that holds its answer. A RAG system also meets
questions it must refuse and questions whose answer depends on a condition, and an evaluation set that leaves them out
cannot tell you how it handles them. Write three records by hand about the Atlas and Boreal policies
(`data/support-case.json`): a paraphrase of the reset-link question that shares few of the policy's words, an
exception (temporary accounts cannot reset by email; H01), and a question no policy answers.

In [11]:
records += [
    {"question_id": "G02", "split": "development", "question": "When does the password recovery email stop working for Atlas?",
     "type": "paraphrase", "known_support": ["H02"], "references": ["30 minutes"], "refuse": False},
    {"question_id": "G05", "split": "development", "question": "Can temporary Atlas accounts use the usual email route?",
     "type": "exception", "known_support": ["H01"], "references": ["support agent"], "refuse": False},
    {"question_id": "G04", "split": "development", "question": "What is tomorrow's weather in Amman?",
     "type": "out_of_scope", "known_support": [], "references": [], "refuse": True},
]
print(len(records), "records |", sum(r["refuse"] for r in records), "to refuse")

10 records | 1 to refuse


The exception's reference is *support agent*, not *no*: the customer needs the route that works, and *no* alone is
found in many wrong answers. G02, G05 and G04 are the IDs these questions have in the course golden set of Part F.

A set you score against must not change unnoticed. **Version** it: a hash of the records changes when any record
does, so a saved result names the exact set it was scored on. Save the records with that hash, the stamp, and hashes
of the answer rules and the data, which also decide the scores:

In [12]:
import hashlib
from shared.answer import RULES

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

data_text = "".join((DATA / name).read_text(encoding="utf-8") for name in ("squad-chunks.json", "support-case.json"))
version = {"records": sha(json.dumps(records, sort_keys=True)), "rules": sha(RULES), "data": sha(data_text),
           **stamp(settings)}
path = ROOT / ".cache" / f"golden-{version['records']}.json"
path.parent.mkdir(exist_ok=True)
path.write_text(json.dumps({"version": version, "records": records}, indent=1, ensure_ascii=False), encoding="utf-8")
print(path.relative_to(ROOT), "|", {name: version[name] for name in ("records", "rules", "data")})

.cache/golden-b5789bcc0fdd.json | {'records': 'b5789bcc0fdd', 'rules': '4be8e10f41b6', 'data': '5d58158d0880'}


The file goes to `.cache/`, outside the index. A golden record inside the searchable corpus would be retrieved and
quoted by the pipeline it is meant to test.

### Checkpoint C

| You should see | Our run |
|---|---|
| The Tesla record | `references: ['1856']`, `known_support: ['S03-P00']`, `refuse: False` |
| Ten records | seven from the development questions, three policy records, 1 to refuse |
| Saved | `.cache/golden-b5789bcc0fdd.json` |

---

## Part D: Run the pipeline on the ten records

Part D produces one row per record: where the support was retrieved, what the model answered, and each measure.

The pipeline is Lab 05's, from `shared/lc_pipeline.py`: `build_store()` embeds the 277 documents into the vector
store, `retrieve()` returns the top three, `evidence()` writes them as `[ID] text` lines, and `answer()` returns a
`CitedAnswer`. Build the store once:

In [13]:
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
print(len(store.store), "documents in the store")

277 documents in the store


Score one record by hand, the paraphrase G02. The store returns documents whose IDs are chunk IDs; the record's
support is in source IDs, so read `source_id` from each document's metadata before you look for the rank.

In [14]:
record = records[7]
hits = lc_pipeline.retrieve(store, record["question"])
sources = [document.metadata["source_id"] for document, _ in hits]
reply = lc_pipeline.answer(record["question"], lc_pipeline.evidence(hits), settings)
print("retrieved:", sources, "| first support at rank", first_rank(sources, record["known_support"]))
print("answer:", reply.answer, "| EM, F1:", em_f1(reply.answer, record["references"]))

retrieved: ['H01', 'H04', 'H02'] | first support at rank 3
answer: The password recovery email for Atlas stops working after 30 minutes. | EM, F1: (0, 0.33333333333333337)


H02, the policy that answers, comes third, behind H01 and the Boreal policy H04: a hit at 3, with a reciprocal
rank of 1/3. The answer is right, *The password recovery email for Atlas stops working after 30 minutes.*, and
scores F1 0.333 (printed in full as 0.33333333333333337): its two reference words are two of its ten.

Now the same steps for every record, as one function that returns a row. A refusal record gets no EM or F1, because
there is no answer to compare with; its row records whether the model said *Insufficient evidence.*

In [15]:
INSUFFICIENT = "Insufficient evidence."

def score_row(record):
    hits = lc_pipeline.retrieve(store, record["question"])
    reply = lc_pipeline.answer(record["question"], lc_pipeline.evidence(hits), settings)
    sources = [document.metadata["source_id"] for document, _ in hits]
    row = {"id": record["question_id"], "type": record["type"], "refuse": record["refuse"],
           "rank": first_rank(sources, record["known_support"]),
           "answer": reply.answer, "refused": reply.answer.strip() == INSUFFICIENT}
    if not record["refuse"]:
        row["em"], row["f1"] = em_f1(reply.answer, record["references"])
    return row

rows = [score_row(record) for record in records]
for record, row in zip(records, rows):
    scores = f"EM {row['em']}  F1 {row['f1']:.3f}" if not row["refuse"] else "refusal record"
    print(f"{record['question'][:36]:38}rank {row['rank']!s:5}{scores:17}| {row['answer']}")

In what year was Nikola Tesla born?   rank 1    EM 0  F1 0.250   | Nikola Tesla was born in the year 1856.
In what year did Nikola Tesla emigra  rank 1    EM 1  F1 1.000   | 1884
Who was the Super Bowl 50 MVP?        rank 2    EM 0  F1 0.444   | Von Miller was the Super Bowl 50 MVP.
How many appearances have the Denver  rank 1    EM 0  F1 0.200   | The Denver Broncos have made eight appearances in the Super Bowl.
What animals does the Vistula river'  rank 1    EM 0  F1 0.778   | The Vistula river's ecosystem includes the otter, beaver, and hundreds of bird species.
What is the original meaning of the   rank 1    EM 0  F1 0.333   | The original meaning of the word Norman is "Northman" or "Norseman, Viking."
What measure of a computational prob  rank 1    EM 0  F1 0.000   | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
When does the password recovery emai  rank 3    EM 0  F1 0.333   | The password recovery email for Atlas stops workin

One row per record, each with the rank of its first support, its answer and its scores. Part E reads them.

### Checkpoint D

| You should see | Our run |
|---|---|
| The store | 277 documents |
| G02 by hand | retrieved `['H01', 'H04', 'H02']`; first support at rank 3; EM 0, F1 0.333 |
| Ten rows | ten rows; rank 1 except the MVP (2), G02 (3) and G04 (`None`); EM 1 only for *1884*; G04 *Insufficient evidence.* |

---

## Part E: Report each measure with its denominator (self-study)

Part E turns the rows into the few numbers you would report, and checks each number against the rows.

Retrieval measures need known support. G04 has none, since no source answers it, so it stays out of hit@3 and MRR,
and the line says so:

In [16]:
with_support = [row for row, record in zip(rows, records) if record["known_support"]]
hits3 = sum(row["rank"] is not None for row in with_support)
mrr = sum(1 / row["rank"] if row["rank"] else 0 for row in with_support) / len(with_support)
print(f"hit@3: {hits3} of {len(with_support)} records with known support ({len(rows) - len(with_support)} without, not counted)")
print(f"MRR@3: {mrr:.3f} over the same {len(with_support)}")

hit@3: 9 of 9 records with known support (1 without, not counted)
MRR@3: 0.870 over the same 9


Every record with support found it in the top three: hit@3 9 of 9. MRR@3 is 0.870, lower, because two records
found it later: the MVP at rank 2 and G02 at rank 3. hit@3 reads as perfect retrieval; MRR shows where it is
not.

The answer measures cover the nine records that expect an answer:

In [17]:
answer_rows = [row for row in rows if not row["refuse"]]
print(f"EM: {sum(row['em'] for row in answer_rows)} of {len(answer_rows)} answer records")
print(f"F1: {sum(row['f1'] for row in answer_rows) / len(answer_rows):.3f}, the mean over the same {len(answer_rows)}")

EM: 1 of 9 answer records
F1: 0.371, the mean over the same 9


Before you trust either number, read the rows it averages, from the lowest F1 up:

In [18]:
for row in sorted(answer_rows, key=lambda row: row["f1"]):
    print(f"F1 {row['f1']:.3f}  rank {row['rank']}  | {row['answer']}")

F1 0.000  rank 1  | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
F1 0.000  rank 1  | No, temporary Atlas accounts cannot use the usual email route.
F1 0.200  rank 1  | The Denver Broncos have made eight appearances in the Super Bowl.
F1 0.250  rank 1  | Nikola Tesla was born in the year 1856.
F1 0.333  rank 1  | The original meaning of the word Norman is "Northman" or "Norseman, Viking."
F1 0.333  rank 3  | The password recovery email for Atlas stops working after 30 minutes.
F1 0.444  rank 2  | Von Miller was the Super Bowl 50 MVP.
F1 0.778  rank 1  | The Vistula river's ecosystem includes the otter, beaver, and hundreds of bird species.
F1 1.000  rank 1  | 1884


The mean F1 of 0.371 reads like a pipeline that is mostly wrong. The rows say otherwise: seven of the nine
answers are right, scoring 0.200 to 1.000 mostly by their length; only *1884*, a bare year, matches exactly.
The two at 0.000 are wrong: the complexity answer names the field instead of the measure, and G05 says *No*
without the support agent a temporary-account customer needs. Both have their support at rank 1, so hit@3 and
MRR count them as perfect: a retrieval measure cannot see the answer.

A check that fits a generated sentence asks whether a whole accepted answer appears in it. Benchmarks of long
generated answers do this: ALCE, for example, checks whether each short reference answer is a substring of the
generated text. `found()` compares whole words after Part A's normalisation; the spaces stop *no* from being found
inside *know*:

In [19]:
def found(prediction, references):
    answer_text = f" {normalize(prediction)} "
    return any(f" {normalize(reference)} " in answer_text for reference in references)

for row, record in zip(rows, records):
    if not record["refuse"]:
        row["found"] = found(row["answer"], record["references"])
print(f"found: {sum(row['found'] for row in answer_rows)} of {len(answer_rows)} answer records")
for row in answer_rows:
    if not row["found"]:
        print("  not found:", row["id"], "|", row["answer"])
print("both products' times, against ['10 minutes']:", found("Atlas: 30 minutes. Boreal: 10 minutes.", ["10 minutes"]))

found: 7 of 9 answer records
  not found: 56e16839cd28a01900c67887 | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
  not found: G05 | No, temporary Atlas accounts cannot use the usual email route.
both products' times, against ['10 minutes']: True


7 of 9, and the two not found are the two wrong answers: the complexity question (shown by its SQuAD ID) and
G05. On these rows the reference check agrees with a reader where F1 could only rank. `found()` has a blind spot of its own, in the last line: it passes an answer that holds the
reference and something else beside it, such as both products' reset times. Lab 07 adds a judge that reads the
claims.

Last, refusals. They get their own lines, with their own denominators, both ways round:

In [20]:
from shared.settings import chat_url

refusal_rows = [row for row in rows if row["refuse"]]
print(f"refused when it should:        {sum(row['refused'] for row in refusal_rows)} of {len(refusal_rows)} refusal records")
print(f"refused when it should answer: {sum(row['refused'] for row in answer_rows)} of {len(answer_rows)} answer records")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)} | golden records {version['records']}")

refused when it should:        1 of 1 refusal records
refused when it should answer: 0 of 9 answer records

gpt-4o-mini at https://api.openai.com/v1 | golden records b5789bcc0fdd


The weather question was refused (1 of 1), and no answer record was refused (0 of 9). One record says little about refusals; Part F scores a set with more.

### Checkpoint E

| You should see | Our run |
|---|---|
| Retrieval | hit@3 9 of 9 (1 without support, not counted); MRR@3 0.870 |
| Answers | EM 1 of 9; F1 0.371; at 0.000 the complexity answer and G05's *No, temporary Atlas accounts ...* |
| `found()` | 7 of 9; not found: the complexity question and G05; both products' times `True` |
| Refusals | refused 1 of 1 refusal records, 0 of 9 answer records |

---

## Part F: Score the course golden set by slice (self-study)

Part F scores the golden set the later labs use, and reports each kind of question, each **slice**, on its own.

The course golden set is the 30 development questions as records, plus eight policy cases in
`06_Evaluation_Set/resources/golden-policy-cases.json`, reviewed from a draft. The review made the changes you would
look for in any golden set someone hands you:

| Draft case | Change at review | Why |
|---|---|---|
| G01, G02, G05 | a described answer (*30 minutes with a supported citation*) became `references` | a check needs text to compare |
| G03 (temporary accounts) | `pack: ["H02"]`: the answer step gets H02 alone | the draft's `evidence_ids` meant the evidence given here and the support elsewhere; retrieval would find H01, which answers it |
| G06 (follow-up) | left out | a follow-up needs the conversation itself (Lab 09) |
| G07, G08, G09 | added | no case was missing from the corpus, and none had the other product's policy ranked first |

`shared/evals.py` holds Parts B to D as functions, reads the golden set, and adds the version and the `pack` case.
Read it:

In [21]:
import inspect
from shared import evals

print(inspect.getsource(evals))

"""Golden records and the measures that score a RAG pipeline on them: the steps Lab 06 builds by hand.

em_f1() and found() compare an answer with its reference answers; first_rank() finds the first retrieved source a
record names as support. record_from_dev() turns a development question into a golden record, and golden_set() reads
the whole development golden set: the 30 development questions and the reviewed policy cases in Lab 06's resources/.
score() runs one record through shared.lc_pipeline and returns its row; version() names the set, the rules and the
data a run used.
"""
from __future__ import annotations

import hashlib
import json
import re
import string
from collections import Counter

from shared.answer import RULES
from shared.lc_pipeline import answer, evidence, retrieve
from shared.settings import DATA, ROOT, stamp

REFUSAL = "Insufficient evidence."
POLICY_CASES = ROOT / "06_Evaluation_Set" / "resources" / "golden-policy-cases.json"


def normalize(text: str) -> str:


`score()` is `score_row()` with two additions: a record with a `pack` is answered from those documents instead of
the search, and every row gets `passed`. A refusal record passes when the model refused; an answer record passes when
`found()` finds a reference. Score all of it:

In [22]:
golden = evals.golden_set()
golden_rows = [evals.score(record, store, settings) for record in golden]
policy_rows = [row for row in golden_rows if row["question_id"].startswith("G")]
print(f"passed {sum(row['passed'] for row in golden_rows)} of {len(golden_rows)} records")
print(f"  the 30 development questions: {sum(row['passed'] for row in golden_rows[:30])} of 30")
print(f"  the {len(policy_rows)} policy cases:          {sum(row['passed'] for row in policy_rows)} of {len(policy_rows)}")

passed 34 of 38 records
  the 30 development questions: 29 of 30
  the 8 policy cases:          5 of 8


34 of 38 looks like a good pipeline, and on the development questions it is: 29 of 30. The policy cases, the
only records written to test refusals and conditions, pass 5 of 8, and in the whole-set count they are 8
records of 38. Split the policy cases by type, and add the refusal slice, every record that should be refused:

In [23]:
for kind in dict.fromkeys(row["type"] for row in policy_rows):
    kind_rows = [row for row in policy_rows if row["type"] == kind]
    print(f"{kind:22}{sum(row['passed'] for row in kind_rows)} of {len(kind_rows)}")
refusal_slice = [row for row in golden_rows if row["refuse"]]
print(f"\nrefusal slice: refused {sum(row['refused'] for row in refusal_slice)} of {len(refusal_slice)}")
for row in golden_rows:
    if not row["passed"]:
        print(f"\nfailed {row['question_id']} ({row['type']}) | retrieved {row['retrieved']} | rank {row['rank']}")
        print("  ", row["answer"])

answerable            1 of 1
paraphrase            1 of 1
missing_pack_support  1 of 1
out_of_scope          1 of 1
exception             0 of 1
missing_from_corpus   1 of 1
wrong_product         0 of 2

refusal slice: refused 3 of 4

failed 56e16839cd28a01900c67887 (answerable) | retrieved ['S04-P01-C00', 'S04-P00-C00', 'S04-P02-C01'] | rank 1
   Computational complexity theory broadly defines the inherent difficulty of a computational problem.

failed G05 (exception) | retrieved ['H01', 'H02', 'H04'] | rank 1
   No, temporary Atlas accounts cannot use the usual email route.

failed G08 (wrong_product) | retrieved ['H03', 'H05', 'S38-P00-C01'] | rank None
   Only workspace owners can export all reports in Boreal.

failed G09 (wrong_product) | retrieved ['H04', 'H01', 'H02'] | rank 2
   Yes, password reset uses a mobile app.


The failed slices are policy slices of one or two records: the exception 0 of 1, the wrong product 0 of 2, the
refusal slice 3 of 4. G08 asks about Boreal; the only export policy is Atlas's (H03, first), and the model
answered *Only workspace owners can export all reports in Boreal.*, the Atlas rule with the name changed. With
no reference and no support, neither F1 nor hit@3 can see it; only the refusal slice does. G09 retrieved the
Boreal policy H04 before the Atlas policy H01 and answered with the Boreal rule, and hit@3 counts it a hit.

Your ten records had one refusal record, and it passed. The failing slices are ones you had not written: what
a golden set can find depends on which kinds of question it holds, not on how many.

A slice of one or two records needs more than one run before you build on it. Score the eight policy cases twice
more and count, for each, the runs it passed:

In [24]:
policy_cases = [record for record in golden if record["question_id"].startswith("G")]
passes = {row["question_id"]: int(row["passed"]) for row in policy_rows}
for _ in range(2):
    for record in policy_cases:
        passes[record["question_id"]] += evals.score(record, store, settings)["passed"]
print("passed, of 3 runs:", passes)

passed, of 3 runs: {'G01': 3, 'G02': 3, 'G03': 3, 'G04': 3, 'G05': 0, 'G07': 3, 'G08': 0, 'G09': 0}


The same three cases failed in all three runs, G05, G08 and G09 (0 of 3 each), and the other five passed 3 of
3. These failures are steady enough to test a change against.

Read these counts for what they are. The 38 records are 30 questions from five Wikipedia articles and eight
cases about five two-sentence policies, and `known_support` names the paragraph each question was written from,
not every passage that could answer it. The counts compare versions of this pipeline on the same records; they
do not estimate how often it will be right on the questions people actually ask.

Save the rows with the set's version, so that a later lab compares a change with this run on the same records and
knows which model and endpoint produced it:

In [25]:
golden_version = evals.version(golden, settings)
model = settings["CHAT_MODEL"].replace("/", "_")
path = ROOT / ".cache" / f"golden-rows-{model}-{golden_version['records']}.json"
path.write_text(json.dumps({"version": golden_version, "rows": golden_rows}, indent=1, ensure_ascii=False),
                encoding="utf-8")
print(path.relative_to(ROOT), "|", len(golden_rows), "rows")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

.cache/golden-rows-gpt-4o-mini-42126604e98f.json | 38 rows

gpt-4o-mini at https://api.openai.com/v1


Labs 07 and 11 load this set with `evals.golden_set()` and score a change with `evals.score()`; Lab 11 turns the
refusal slice into a release test that no average can pass.

### Checkpoint F

| You should see | Our run |
|---|---|
| Whole set | passed 34 of 38; development questions 29 of 30; policy cases 5 of 8 |
| Slices | exception 0 of 1, wrong_product 0 of 2, the other types 1 of 1; refusal slice 3 of 4; failed: the complexity question, G05, G08, G09 |
| Three runs | G05, G08 and G09 0 of 3; the other five 3 of 3 |
| Saved | `.cache/golden-rows-gpt-4o-mini-42126604e98f.json`, 38 rows |

---

## What you can do now

- Compute hit@3 and MRR@3 with every question in the denominator, and say which records a retrieval measure cannot
  count.
- Compute EM and token F1 against several references, and say why a right sentence can score below a wrong short answer.
- Write golden records with a type, known support, and references or an expected refusal, and version a set with
  hashes and the stamp.
- Run a pipeline on a golden set and keep one row per question, with what was retrieved and answered.
- Report refusals and each kind of question as their own slices, and read the failing rows behind every count.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Write records for a slice the set lacks.** Under **Task 1**, complete two records before you run anything:
   *Where do Boreal workspace owners download invoices?* (no Boreal billing policy exists) and *How long is a Boreal
   reset link valid?* (H04). The cell scores each three times with `evals.score()` and prints the answers.

   **Check:** for each record, the runs passed and the answers. Our worked solution: the billing record passed 0 of 3 (*Workspace owners can download invoices from Billing.*, twice, and once with *Boreal* added: the Atlas billing rule), and the reset record 3 of 3 (*Boreal reset links are valid for 10 minutes.*, support at rank 1).

   **Explain:** which measures from this lab would have counted the billing answer as a success had you written its
   record as answerable, with H05 as support? Which record field decides whether the failure is seen?

2. **Optional: version a change to the set.** Add *half an hour* as a second reference to G01, in a copy of the set.

   **Check:** the records hash before and after, and G01's F1 against the old and new references. Our worked solution: the hash changed from `42126604e98f` to `597edd27f28d`, and G01's F1 stayed 0.444 for *Atlas reset links expire after 30 minutes.*

   **Explain:** why should rows saved under the old hash not be compared with rows scored on the new set, even when
   G01's score did not change?

**Task 1.** Complete the two records, then run the cell:

In [26]:
billing = {"question_id": "C01", "split": "development", "question": "Where do Boreal workspace owners download invoices?",
           "type": None, "known_support": None, "references": None, "refuse": None}   # TODO
reset = {"question_id": "C02", "split": "development", "question": "How long is a Boreal reset link valid?",
         "type": None, "known_support": None, "references": None, "refuse": None}       # TODO

if None in billing.values() or None in reset.values():
    print("⚠️ Complete both records, then run the cell again.")
else:
    for record in (billing, reset):
        runs = [evals.score(record, store, settings) for _ in range(3)]
        print(f"{record['question_id']} passed {sum(row['passed'] for row in runs)} of 3 | rank {runs[0]['rank']}")
        for row in runs:
            print("  ", row["answer"])

⚠️ Complete both records, then run the cell again.


**Task 2 (optional).** Add the reference to G01 in `changed`, then run the cell:

In [27]:
import copy

changed = copy.deepcopy(golden)
g01 = next(record for record in changed if record["question_id"] == "G01")
# TODO: add "half an hour" to g01["references"]

if g01["references"] == ["30 minutes"]:
    print("⚠️ Add the reference, then run the cell again.")
else:
    print("records hash before:", evals.version(golden, settings)["records"],
          "| after:", evals.version(changed, settings)["records"])
    answer_text = next(row["answer"] for row in golden_rows if row["question_id"] == "G01")
    print("G01 F1, old references:", round(evals.em_f1(answer_text, ["30 minutes"])[1], 3),
          "| new references:", round(evals.em_f1(answer_text, g01["references"])[1], 3), "|", answer_text)

⚠️ Add the reference, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ZeroDivisionError: division by zero` in an F1 function | precision divided by the answer's length before checking the overlap, on an empty answer | Test `if overlap` before dividing, as `em_f1()` does |
| hit@3 is 0 for every development record | the documents' IDs (chunk IDs, `S03-P00-C00`) compared with `known_support` (source IDs, `S03-P00`) | Compare `document.metadata["source_id"]` |
| `TypeError: unsupported operand type(s) for /: 'int' and 'NoneType'` | `1 / row["rank"]` on a record whose support was not retrieved | Write `1 / rank if rank else 0`, and keep the record in the denominator |
| `KeyError: 'f1'` | an average over every row, refusal records included | Average over the answer records only, and give refusals their own line |

## Resources

- [Rajpurkar et al., *SQuAD: 100,000+ Questions for Machine Comprehension of Text*, 2016](https://arxiv.org/abs/1606.05250):
  the dataset, its several answers per question, and the EM and F1 its scoring script computes
- [Gao et al., *Enabling Large Language Models to Generate Text with Citations* (ALCE), 2023](https://arxiv.org/abs/2305.14627):
  checking short reference answers inside long generated answers, and citation quality
- [LangSmith: evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts): datasets of examples,
  reference outputs, evaluators and experiments, the same ideas as a hosted service
- [Mean reciprocal rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank): the definition, with a worked example

**The same idea in three RAG books' code**

- [*Hands-On RAG for Production*, `chapter6/metrics.ipynb`](https://github.com/ofermend/hands-on-rag/blob/main/chapter6/metrics.ipynb): context precision and recall from retrieved and
  relevant chunk IDs, and BLEU and ROUGE-L for the answers, on three worked examples
- [*RAG from First Principles*, `09-Evaluation/`](https://github.com/PacktPublishing/RAG-from-First-Principles/tree/main/09-Evaluation): one script each for RAGAS, TruLens, DeepEval and LlamaIndex's
  evaluators
- [*Unlocking Data with Generative AI and RAG* (2nd ed.), `CHAPTER_09/`](https://github.com/PacktPublishing/Unlocking-Data-with-Generative-AI-and-RAG-Second-Edition/tree/main/CHAPTER_09): RAGAS generating a ten-question test set from a PDF, then
  scoring a vector-search chain against a hybrid one

Next: [Lab 07](../07_LLM_Judge_Calibration/README.md), *How to check an LLM judge against your own labels*. Back to [the lab list](../README.md).